In [ ]:
import pandas as pd
import re
from datetime import datetime



def load_csv(path):
    df = pd.read_csv(path, dtype=str)
    # drop accidental empty unnamed columns
    unnamed = [c for c in df.columns if c.startswith("Unnamed")]
    if unnamed:
        print(f"Dropping unnamed columns from {path}: {unnamed}")
        df = df.drop(columns=unnamed)
    return df

def is_valid_email(email):
    if pd.isna(email):
        return False
    return bool(re.fullmatch(r"[^@]+@[^@]+\.[^@]+", email.strip()))

def is_valid_phone(phone):
    if pd.isna(phone):
        return False
    phone = str(phone).strip()
    return bool(re.fullmatch(r"\d{7,15}", phone))

def as_datetime(series, fmt=None):
    return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)

def report(df, name):
    print(f"\n=== {name} ===")
    print("shape:", df.shape)
    print("missing per column:\n", df.isna().sum())
    print("duplicate rows:", df.duplicated().sum())

members = load_csv("members.csv")
books = load_csv("books.csv")
loans = load_csv("Loans.csv")
bookcopies = load_csv("BookCopies.csv")
staff = load_csv("staff.csv")
tier = pd.DataFrame({
    "TierID": ["T0001", "T0002", "T0003"],
    "TierName": ["Basic", "Advanced", "Premium"],
    "TierLevel": [1, 2, 3],
    "BorrowLimit": [20, 30, 50]
})

report(members, "members")
report(books, "books")
report(loans, "loans")
report(bookcopies, "bookcopies")
report(staff, "staff")

# 1) Business rule / invalid value checks

print("\n--- Value rule checks ---")

def check_allowed(df, column, allowed, df_name):
    bad = df.loc[~df[column].isin(allowed) & df[column].notna(), column].unique()
    if len(bad):
        print(f"{df_name}.{column} invalid values:", bad)
    else:
        print(f"{df_name}.{column} all valid")

check_allowed(members, "Gender", ["Male", "Female"], "members")
check_allowed(members, "TierID", ["T0001", "T0002", "T0003"], "members")
check_allowed(books, "MinTierID", ["T0001", "T0002", "T0003"], "books")
check_allowed(loans, "LoanStatus", ["Returned", "Lost"], "loans")
check_allowed(bookcopies, "Status", ["Available", "Unavailable", "Lost"], "bookcopies")
check_allowed(staff, "Gender", ["Male", "Female"], "staff")
check_allowed(staff, "EmploymentType", ["Full-time", "Part-time"], "staff")

# 2) Referential integrity checks

print("\n--- Referential integrity ---")
def ref_check(child, child_col, parent, parent_col, child_name, parent_name):
    missing = child.loc[~child[child_col].isin(parent[parent_col]), child_col].unique()
    if len(missing):
        print(f"{child_name}.{child_col} values not found in {parent_name}.{parent_col}:", missing[:20])
    else:
        print(f"{child_name}.{child_col} all found in {parent_name}.{parent_col}")

ref_check(members, "TierID", tier, "TierID", "members", "tier")
ref_check(books, "MinTierID", tier, "TierID", "books", "tier")
ref_check(bookcopies, "BookID", books, "BookID", "bookcopies", "books")
ref_check(loans, "MemberID", members, "MemberID", "loans", "members")
ref_check(loans, "BookID", bookcopies, "BookID", "loans", "books")
ref_check(loans, "StaffID", staff, "StaffID", "loans", "staff")
print(loans.loc[~loans[["BookID", "CopySeqNo"]].apply(tuple, axis=1).isin(bookcopies[["BookID", "CopySeqNo"]].apply(tuple, axis=1)), ["BookID", "CopySeqNo"]])

# 3) Key integrity and consistency

print("\n--- Key integrity / duplicates ---")

def key_duplicates(df, cols, name):
    if any(col not in df.columns for col in cols):
        return
    dup = df[df.duplicated(subset=cols, keep=False)].sort_values(cols)
    print(f"{name} duplicates on {cols}: {len(dup)} rows")
    if not dup.empty:
        print(dup.head(10).to_string(index=False))

key_duplicates(members, ["MemberID"], "members")
key_duplicates(books, ["BookID"], "books")
key_duplicates(staff, ["StaffID"], "staff")
key_duplicates(bookcopies, ["BookID", "CopySeqNo"], "bookcopies")
key_duplicates(loans, ["MemberID", "BookID", "CopySeqNo", "LoanDatetime"], "loans")

# 4) Date validity checks

print("\n--- Date checks ---")
members["DateOfBirth"] = as_datetime(members["DateOfBirth"])
members["RegistrationDatetime"] = as_datetime(members["RegistrationDatetime"])
loans["LoanDatetime"] = as_datetime(loans["LoanDatetime"])
loans["DueDate"] = as_datetime(loans["DueDate"])
loans["ReturnDatetime"] = as_datetime(loans["ReturnDatetime"])
staff["StartDatetime"] = as_datetime(staff["StartDatetime"])
staff["DOB"] = as_datetime(staff["DOB"])

print("members DateOfBirth parse failures:", members["DateOfBirth"].isna().sum())
print("members RegistrationDatetime parse failures:", members["RegistrationDatetime"].isna().sum())
print("loans LoanDatetime parse failures:", loans["LoanDatetime"].isna().sum())
print("loans DueDate parse failures:", loans["DueDate"].isna().sum())
print("loans ReturnDatetime parse failures:", loans["ReturnDatetime"].isna().sum())
print("staff StartDatetime parse failures:", staff["StartDatetime"].isna().sum())
print("staff DOB parse failures:", staff["DOB"].isna().sum())

bad_dob = members.loc[members["DateOfBirth"] > pd.Timestamp.today(), ["MemberID", "DateOfBirth"]]
if not bad_dob.empty:
    print("members with future DateOfBirth:")
    print(bad_dob.to_string(index=False))

bad_register = members.loc[
    (members["RegistrationDatetime"].notna()) &
    (members["DateOfBirth"].notna()) &
    (members["RegistrationDatetime"] < members["DateOfBirth"]),
    ["MemberID", "DateOfBirth", "RegistrationDatetime"]
]
if not bad_register.empty:
    print("members registered before DateOfBirth:")
    print(bad_register.to_string(index=False))

bad_due = loans.loc[
    loans["LoanDatetime"].notna() &
    loans["DueDate"].notna() &
    (loans["DueDate"] < loans["LoanDatetime"]),
    ["MemberID", "BookID", "LoanDatetime", "DueDate"]
]
if not bad_due.empty:
    print("loans with DueDate before LoanDatetime:")
    print(bad_due.head(20).to_string(index=False))

bad_return = loans.loc[
    loans["LoanDatetime"].notna() &
    loans["ReturnDatetime"].notna() &
    (loans["ReturnDatetime"] < loans["LoanDatetime"]),
    ["MemberID", "BookID", "LoanDatetime", "ReturnDatetime"]
]
if not bad_return.empty:
    print("loans with ReturnDatetime before LoanDatetime:")
    print(bad_return.head(20).to_string(index=False))

# 5) Format / completeness checks

print("\n--- Format and completeness checks ---")
members_email_bad = members.loc[~members["Email"].apply(is_valid_email)]
print("members bad emails:", len(members_email_bad))
members_phone_bad = members.loc[~members["ContactNo"].apply(is_valid_phone)]
print("members bad ContactNo:", len(members_phone_bad))
staff_email_bad = staff.loc[~staff["Contact"].apply(is_valid_email)]
print("staff bad Contact:", len(staff_email_bad))

for df, name in [(members, "members"), (books, "books"),
                 (loans, "loans"), (bookcopies, "bookcopies"), (staff, "staff")]:
    print(f"{name} missing rows count:", df.isna().any(axis=1).sum())

# 6) Check inconsistent categorical values

print("\n--- Categorical consistency ---")
for df, col, name in [
    (bookcopies, "Status", "bookcopies"),
    (members, "TierID", "members"),
    (books, "MinTierID", "books"),
    (staff, "EmploymentType", "staff")
]:
    if col in df.columns:
        vals = sorted(df[col].dropna().unique())
        print(f"{name}.{col} values:", vals)



Dropping unnamed columns from Loans.csv: ['Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']

=== members ===
shape: (301, 9)
missing per column:
 MemberID                 0
FullName                 0
ContactNo                0
Email                    0
DateOfBirth              0
Gender                   0
Occupation              43
RegistrationDatetime     0
TierID                   0
dtype: int64
duplicate rows: 0

=== books ===
shape: (500, 5)
missing per column:
 BookID       0
Title        0
Author       0
Genre        0
MinTierID    0
dtype: int64
duplicate rows: 0

=== loans ===
shape: (12345, 8)
missing per column:
 MemberID          0
BookID            0
CopySeqNo         0
StaffID           0
LoanDatetime      0
DueDate           0
ReturnDatetime    1
LoanStatus        7
dtype: int64
duplicate rows: 0

=== bookcopies ===
shape: (1265, 3)
missing per column:
 BookID       0
CopySeqNo    0
Status       0
dtype: int64
duplicate rows: 0

=== 

C:\Users\JunHao\AppData\Local\Temp\ipykernel_3788\2113374978.py:28: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
C:\Users\JunHao\AppData\Local\Temp\ipykernel_3788\2113374978.py:28: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(series, errors="coerce", infer_datetime_format=True)
C:\Users\JunHao\AppData\Local\Temp\ipykernel_3788\2113374978.py:28: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict ve

In [85]:
loans.loc[loans["LoanStatus"].isnull()]

,MemberID,BookID,CopySeqNo,StaffID,LoanDatetime,DueDate,ReturnDatetime,LoanStatus
9107,M0017,B00489,2,S0045,2025-12-31 15:31:44,2026-01-14,NaT,NaN
12230,M0201,B00338,5,S0049,2025-12-30 15:17:00,2026-01-13,2026-01-08 16:13:48,NaN
12231,M0077,B00438,1,S0046,2025-12-31 15:53:43,2026-01-14,2026-01-05 09:28:53,NaN
12232,M0035,B00332,2,S0024,2025-12-31 16:02:19,2026-01-14,2026-01-01 10:12:29,NaN
12233,M0049,B00313,1,S0001,2025-12-31 14:58:56,2026-01-14,2026-01-06 10:04:48,NaN
12234,M0272,B00051,6,S0032,2025-12-31 12:29:34,2026-01-14,2026-01-06 13:04:38,NaN
12235,M0008,B00071,1,S0018,2025-12-30 14:44:52,2026-01-13,2026-01-02 09:45:30,NaN


In [86]:
loans.loc[9107, 'LoanStatus'] = "Lost"
loans.loc[loans['LoanStatus'].isnull(), 'LoanStatus'] = "Returned"

In [13]:
print(members.loc[members["TierID"]=="T00001", "TierID"])

196    T00001
Name: TierID, dtype: object


In [87]:
members.loc[members["TierID"]=="T00001", "TierID"] = "T0001"

In [14]:
bookcopies.loc[bookcopies["Status"]=="Availble", "Status"]

1173    Availble
1186    Availble
1223    Availble
1240    Availble
Name: Status, dtype: object

In [88]:
bookcopies.loc[bookcopies["Status"]=="Availble", "Status"] = "Available" 

In [15]:
loans.loc[loans['StaffID'] == "S0101", 'StaffID']

387    S0101
Name: StaffID, dtype: object

In [89]:
loans.loc[loans['StaffID'] == "S0101", 'StaffID'] = "S0001"

In [90]:
import datetime as dt
loans.loc[(loans['BookID'] == "B00153") & (loans['LoanDatetime'] >= "2024-07-04") & (loans['LoanDatetime'] <= "2025-09-04")]
loans.loc[387, ['CopySeqNo', 'LoanStatus', 'DueDate']] = [3, "Returned", (loans.loc[387, "LoanDatetime"] + dt.timedelta(days=14))]
loans.loc[387]

MemberID                        M0107
BookID                         B00153
CopySeqNo                           3
StaffID                         S0001
LoanDatetime      2024-07-04 11:38:15
DueDate           2024-07-18 11:38:15
ReturnDatetime    2025-09-04 16:04:53
LoanStatus                   Returned
Name: 387, dtype: object

In [16]:
members.loc[members["DateOfBirth"].isna(), "DateOfBirth"]

300   NaT
Name: DateOfBirth, dtype: datetime64[ns]

In [91]:
members.loc[members["DateOfBirth"].isna(), "DateOfBirth"] = pd.to_datetime("1990-05-08")

In [17]:
staff.loc[staff['StartDatetime'].isnull(), 'StartDatetime']

27   NaT
Name: StartDatetime, dtype: datetime64[ns]

In [ ]:
staff.loc[staff['StartDatetime'].isnull(), 'StartDatetime'] = pd.to_datetime("2/18/2011 13:55")

StaffID                         S0028
StaffName                Emma Roberts
Gender                         Female
Contact           S0028@greengrass.sg
EmploymentType              Part-time
StartDatetime     2011-02-18 13:55:00
Department           Digital Services
DOB               1988-12-22 00:00:00
Name: 27, dtype: object

In [94]:
def to_csv(df, path):
    df.to_csv("cleaned_" + path, index=False)

to_csv(members, "members.csv")
to_csv(books, "books.csv")
to_csv(bookcopies, "bookcopies.csv")
to_csv(loans, "loans.csv")
to_csv(staff, "staff.csv")